# EasyRAG Query Normalization And Preprocessing

## What you will do

- inspect query normalization, rewrite, and MQE expansion directly
- compare preprocessing toggles on the same question
- verify that `QueryResult.metadata` preserves the prepared query variants

## Why this matters

Retrieval quality does not start at the vector store. EasyRAG first prepares the query, and that preparation affects every later ranking step.


## Source code anchors

- `easyrag.rag.retrieval.preprocess.normalize_query`
- `easyrag.rag.retrieval.preprocess.QueryPreprocessor.prepare`
- `easyrag.rag.types.QueryParam`
- `easyrag.rag.types.QueryResult`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.retrieval.preprocess import QueryPreprocessor, normalize_query


## Deterministic path

We start by isolating preprocessing from retrieval. That makes it easier to understand what each toggle actually changes.


In [ ]:
query_text = "   How does query rewrite help retrieval??   "
preprocessor = QueryPreprocessor(_stub_query_model)

rich_param = QueryParam(mode="hybrid", rewrite_enabled=True, mqe_enabled=True, mqe_variants=2)
minimal_param = QueryParam(mode="hybrid", rewrite_enabled=False, mqe_enabled=False)

rich_preparation = preprocessor.prepare(query_text, rich_param)
minimal_preparation = preprocessor.prepare(query_text, minimal_param)

print("normalize_query output")
print(normalize_query(query_text))
print("\nWith rewrite and MQE")
pprint(rich_preparation)
print("\nWithout rewrite and MQE")
pprint(minimal_preparation)


## Output inspection

The next cell runs a full query so you can see the same prepared values inside `QueryResult.metadata`. This is the handoff point between preprocessing and ranked retrieval.


In [ ]:
query_tmp = tempfile.TemporaryDirectory()
query_rag = EasyRAG(
    working_dir=query_tmp.name,
    workspace="query-prep-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(query_rag.initialize_storages())
run_sync(
    query_rag.ainsert(
        [
            "# Architecture\nEasyRAG uses query rewrite to expand the candidate set before grounded retrieval.\n",
            "# FAQ\nMetadata filters and rerank happen after query preparation.\n",
        ],
        ids=["doc::architecture", "doc::faq"],
        file_paths=["docs/architecture.md", "docs/faq.md"],
    )
)

retrieval_result = run_sync(query_rag.aquery(query_text, rich_param))
metadata_snapshot = {
    key: retrieval_result.metadata[key]
    for key in [
        "original_query",
        "normalized_query",
        "rewritten_query",
        "expanded_queries",
        "retrieval_queries",
        "candidate_counts",
        "stage_timings_ms",
    ]
}
_print_json(metadata_snapshot)


## Provider-backed path

When provider-backed defaults are available, the same query-preparation steps run before retrieval. The cell below uses a real `EasyRAG` instance and prints the prepared-query metadata if the call succeeds.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-query-prep-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                [
                    "# Retrieval\nQuery rewrite expands retrieval coverage.\n",
                    "# Filters\nMetadata filters apply after retrieval candidates are assembled.\n",
                ],
                ids=["doc::retrieval", "doc::filters"],
                file_paths=["docs/retrieval.md", "docs/filters.md"],
            )
        )
        provider_result = run_sync(provider_rag.aquery(query_text, QueryParam(mode="hybrid", chunk_top_k=2)))
        _print_json({k: provider_result.metadata.get(k) for k in ["normalized_query", "rewritten_query", "expanded_queries", "retrieval_queries"]})
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

The original question never disappears. EasyRAG keeps the raw string, the normalized form, the rewritten query, the MQE variants, and the final retrieval-query list side by side. That trace makes it much easier to debug when retrieval looks surprising.


In [ ]:
run_sync(query_rag.finalize_storages())
query_tmp.cleanup()
print("Cleaned up the deterministic query-preprocessing workspace.")


## Next steps

- Continue with [04_04_hybrid_metadata_filter_and_modes.ipynb](04_04_hybrid_metadata_filter_and_modes.ipynb) to see how those prepared queries interact with retrieval modes and metadata filters.
- Read [04-retrieval-overview.md](../../docs/04-retrieval-overview.md) for the chapter-level explanation of retrieval preprocessing and ranking.
- Keep [00_02_observing_rag_artifacts.ipynb](../00_overview/00_02_observing_rag_artifacts.ipynb) in mind if you want the artifact-oriented view of the same metadata.
